# NB10: TSS batch effects (L2 norm stability across 710 sites)

Aggregates the per-site embedding statistics emitted by NB03 to reproduce
Supplementary Table S10 and Figure 4D. Quantifies whether OpenCLIP
embedding magnitude is stable across the 710 TCGA tissue source sites.

**Note on the GitHub monolith**: there is no global TSS aggregation
script in `notebook.py`. The per-site stats parquet written by NB03
(`embeddings/per_site_emb_stats.parquet`) is the input to this analysis.

**Manuscript reference** (Supp Table S10, Discussion):
- Total TSS sites: 710
- Mean of site-mean L2 norms: 13.503
- Standard deviation: 0.287
- Range: 12.97 - 14.04
- Coefficient of variation: 2.1%

Low CV = limited site-driven magnitude shifts in OpenCLIP feature norms,
consistent with the manuscript's claim that batch effects are controlled.

**Required env**: `WORKSPACE`. **Inputs**: `embeddings/per_site_emb_stats.parquet`.
**Outputs**: `results/tss_batch/{site_stats.csv, summary.json}`,
`figures/fig4d_l2_distribution.{png,pdf}`.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
EMB_DIR = WORKSPACE / "embeddings"
FIG_DIR = WORKSPACE / "figures"
RESULTS_DIR = WORKSPACE / "results" / "tss_batch"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"WORKSPACE: {WORKSPACE}")


In [ ]:
# load per-site stats from NB03
src = EMB_DIR / "per_site_emb_stats.parquet"
assert src.exists(), f"missing {src}; run NB03 first"
sites = pd.read_parquet(src)
sites["site"] = sites["site"].astype(str)
print(f"sites: {len(sites)}  | columns: {list(sites.columns)}")
print(sites.head(5).to_string(index=False))


In [ ]:
# global aggregation across sites
norms = sites["mean_norm"].astype(float).values
mu = float(np.mean(norms))
sd = float(np.std(norms, ddof=0))
mn = float(np.min(norms))
mx = float(np.max(norms))
cv = sd / mu if mu else float("nan")

print(f"sites              : {len(norms)}        (manuscript ref: 710)")
print(f"mean L2 norm       : {mu:.3f}            (manuscript ref: 13.503)")
print(f"std                : {sd:.3f}            (manuscript ref: 0.287)")
print(f"range              : [{mn:.3f}, {mx:.3f}]  (manuscript ref: [12.97, 14.04])")
print(f"coefficient of var.: {cv*100:.2f}%        (manuscript ref: 2.1%)")


In [ ]:
# write Supp Table S10 csv + summary json
sites_out = sites.sort_values("mean_norm").reset_index(drop=True)
sites_out.to_csv(RESULTS_DIR / "site_stats.csv", index=False)

summary = {
    "n_sites": int(len(norms)),
    "mean_L2": round(mu, 4),
    "std_L2": round(sd, 4),
    "min_L2": round(mn, 4),
    "max_L2": round(mx, 4),
    "cv_pct": round(100 * cv, 3),
    "manuscript_refs": {
        "n_sites": 710, "mean_L2": 13.503, "std_L2": 0.287,
        "range_L2": [12.97, 14.04], "cv_pct": 2.1,
    },
}
(RESULTS_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))


In [ ]:
# Figure 4D: distribution of site-mean L2 norms across 710 TSS sites
fig, ax = plt.subplots(figsize=(5.0, 3.6))
ax.hist(norms, bins=40, color="C0", edgecolor="black", lw=0.4, alpha=0.85)
ax.axvline(mu, color="C3", lw=1.2, ls="--", label=f"mean = {mu:.2f}")
ax.axvspan(mu - sd, mu + sd, color="C3", alpha=0.10, label=f"+/- 1 SD ({sd:.2f})")
ax.set_xlabel("Site-mean L2 norm of OpenCLIP embeddings")
ax.set_ylabel("Number of TSS sites")
ax.set_title(f"Figure 4D: L2 norm stability across {len(norms)} TSS sites (CV = {cv*100:.1f}%)")
ax.legend(loc="upper right", fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4d_l2_distribution.png", dpi=300)
fig.savefig(FIG_DIR / "fig4d_l2_distribution.pdf")
plt.show()
